# GHG Emissions - Raw Data Ingestion

## Objective

This notebook retrieves the GHG emissions dataset from the Paris OpenData API
and stores the raw JSON response in a Databricks Volume.

### Data Flow

Paris OpenData API
→ Raw JSON
→ Databricks Volume

### Source

Paris OpenData dataset:
Inventaire des émissions de gaz à effet de serre du territoire

### Storage Location

/Volumes/workspace/default/raw_data/ghg_emissions_raw.json

### Processing

No transformations are applied at this stage.
The objective is to preserve the source data in its raw form.

In [0]:
import os
import requests
import json


# 1. Define Target Endpoint & Volume Path
API_URL = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets/inventaire-des-emissions-de-gaz-a-effet-de-serre-du-territoire/exports/json"
VOLUME_PATH = "/Volumes/workspace/default/raw_data"
FILE_NAME = "ghg_emissions_raw.json"
TARGET_FILE_PATH = os.path.join(VOLUME_PATH, FILE_NAME)

# 2. Ensure Volume Directory Exists
os.makedirs(VOLUME_PATH, exist_ok=True)

# 3. Fetch Raw Data from OpenData Paris API
print(f"Fetching raw API payload from OpenData Paris...")
response = requests.get(API_URL)
response.raise_for_status()

# 4. Land Raw JSON Payload Directly to Volume
print(f"Landing raw file to Volume at: {TARGET_FILE_PATH}...")
with open(TARGET_FILE_PATH, "wb") as f:
    f.write(response.content)

print(f"Successfully landed raw API response to volume path!")

In [0]:

with open("/Volumes/workspace/default/raw_data/ghg_emissions_raw.json", "r") as f:
    data = json.load(f)

print(data[0].keys())


# Read original JSON to get the original column order
with open("/Volumes/workspace/default/raw_data/ghg_emissions_raw.json", "r") as f:
    data = json.load(f)

original_columns = list(data[0].keys())

# Read JSON with Spark
df = spark.read.option("multiline", "true").json(
    "/Volumes/workspace/default/raw_data/ghg_emissions_raw.json"
)



In [0]:
# Reorder columns according to the original JSON and display the dataframe
df = df.select(*original_columns)

display(df)